# SINASC — auditoria histórica para a extensão temporal (06)

**Referência certificada:** SINASC 2024 (notebooks 01–05, commit `a3c9095`), congelada.
**Extensão em desenvolvimento:** análise multianual. Nada neste notebook é causalmente validado.

## Objetivo

Decidir, com evidência de metadados oficiais e dos próprios arquivos, **quais anos do SINASC podem receber o mesmo desenho de 2024** e com quais riscos. A pergunta desta fase não é "quanto mais anos melhor", e sim se o padrão de 2024 é estável ou particular daquele ano.

Acrescentar anos **não** aumenta automaticamente a validade causal: mais registros não resolvem confundimento residual, seleção entre nascidos vivos nem mudanças de registro ao longo do tempo.

## Escopo desta rodada

- inspeção de disponibilidade e de esquema de 1996 a 2026, sem baixar os arquivos completos;
- comparação das variáveis críticas e de seus códigos;
- proposta de janela temporal e critérios de admissão de um ano.

Não há estimação de efeito, modelos, AIPW multianual nem análise empilhada (*pooled*). Os notebooks 01–05 não são lidos nem alterados, apenas os artefatos agregados que certificam 2024.

In [1]:
from pathlib import Path
from datetime import date
import sys, json, hashlib
import pandas as pd
from IPython.display import display
raiz = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / 'src').exists()).resolve()
sys.path.insert(0, str(raiz))
assert Path(sys.executable).resolve() == (raiz / '.venv/Scripts/python.exe').resolve(), 'Use a .venv do projeto'
from src import temporal as tp
from src.sinasc import calcular_sha256

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
pasta_temporal = raiz / 'outputs' / 'temporal'
cache_inspecao = pasta_temporal / 'inspecao_schema_sinasc.json'
REATUALIZAR_INSPECAO = False  # True refaz as requisições HTTP Range (~47 MB no total)
ANOS_CANDIDATOS = list(range(tp.ANO_MINIMO_PUBLICADO, 2027))
print('Raiz do projeto:', raiz)

Raiz do projeto: C:\GitHub\data-science-projects\projetos\big-data-sinasc-causal-ml


## 1. Contrato da referência certificada 2024

O que **não pode mudar** nesta frente:

| Elemento | Valor congelado |
|---|---|
| População | nascidos vivos SINASC 2024, `MESPRENAT` 1–9, peso 500–6.000 g, gestação única (`GRAVIDEZ=1`) |
| N principal | 2.251.570 (T=1: 1.942.045; T=0: 309.525) |
| Tratamento | início do pré-natal nos meses 1–3 (T=1) versus 4–9 (T=0); 99/vazio fora |
| Desfecho | baixo peso: `PESO` < 2.500 g |
| Covariáveis | idade, escolaridade 2010, raça/cor, situação conjugal, paridade, perdas fetais e UF de residência da mãe |
| AIPW C1 / C2 | −1,340526785646844 pp / −1,2314558359825436 pp |
| Fase 3 | `GATE_FASE2_EXCESSIVAMENTE_CONSERVADOR` |
| Fase 4 | CATE HGB médio −1,294100493864875 pp; 84,68% negativos; `HETEROGENEIDADE_SENSIVEL_A_MODELO` |
| Arquivos | `data/raw/SINASC_2024_csv.zip`, `data/processed/sinasc_2024.parquet`, caches `.npz`, notebooks 01–05 |

A célula abaixo confere esses números nos artefatos versionados e os hashes dos arquivos locais. A extensão temporal lê 2024 **somente** por esses caminhos legados (`tp.caminhos_ano(raiz, 2024)`), sem migrar nem reescrever nada.

In [2]:
ler = lambda nome: json.loads((raiz / 'outputs/diagnostics' / nome).read_text(encoding='utf-8'))
aipw = {r['especificacao']: r['estimativa_pp'] for r in ler('fase2_aipw.json')['resultados'] if r['populacao'] == 'sem_trimming'}
fase4 = ler('fase4_resultados.json')
contrato = {
    'N principal': ler('fase1_amostra.json')['fluxo_amostra'][-1]['n_restante'],
    'AIPW C1 (pp)': aipw['C1'],
    'AIPW C2 (pp)': aipw['C2'],
    'Gate Fase 3': ler('fase3_gate.json')['gate'],
    'CATE HGB médio (pp)': fase4['principal']['media_pp'],
    'CATE negativo (%)': 100 * fase4['principal']['proporcao_negativa'],
    'Gate Fase 4': fase4['gate'],
}
assert contrato['N principal'] == 2251570
assert contrato['AIPW C1 (pp)'] == -1.340526785646844 and contrato['AIPW C2 (pp)'] == -1.2314558359825436
assert contrato['Gate Fase 3'] == 'GATE_FASE2_EXCESSIVAMENTE_CONSERVADOR'
assert contrato['CATE HGB médio (pp)'] == -1.294100493864875
assert contrato['Gate Fase 4'] == 'HETEROGENEIDADE_SENSIVEL_A_MODELO'

c2024 = tp.caminhos_ano(raiz, 2024)
hash_zip = tp.baixar_sinasc_ano(raiz, 2024)['sha256']            # só verifica; não baixa nem escreve
hash_parquet = calcular_sha256(c2024.parquet)
assert hash_parquet == ler('fase2_aipw.json')['provenance']['parquet_sha256']
contrato['SHA-256 ZIP 2024'] = hash_zip
contrato['SHA-256 Parquet 2024'] = hash_parquet
display(pd.Series(contrato, name='valor certificado').to_frame())

,valor certificado
N principal,2251570
AIPW C1 (pp),-1.340527
AIPW C2 (pp),-1.231456
Gate Fase 3,GATE_FASE2_EXCESSIVAMENTE_CONSERVADOR
CATE HGB médio (pp),-1.2941
CATE negativo (%),84.684198
Gate Fase 4,HETEROGENEIDADE_SENSIVEL_A_MODELO
SHA-256 ZIP 2024,27e55de6359e1b1914301dc1d752d78e9535a7a52cc064...
SHA-256 Parquet 2024,a7a1618e555d1417f692b5334b2a0fd01cfee911b3d45c...


## 2. Disponibilidade oficial por ano

Fonte: recurso "Nascidos Vivos - <ano>" do Portal de Dados Abertos do SUS, no mesmo padrão de URL do arquivo certificado de 2024.

Método, sem baixar os ZIPs completos (≈2,5 GB):

1. requisição `HEAD` para existência, tamanho e última modificação;
2. leitura do diretório central do ZIP e de ~1,5 MB comprimidos do início do CSV via HTTP Range;
3. do prefixo descomprimido: nome do CSV interno, encoding, separador, cabeçalho e códigos observados das variáveis críticas.

A inspeção foi validada contra o ZIP local de 2024: mesmo CSV interno, encoding, separador, 62 colunas na mesma ordem e mesmo tamanho descomprimido (561.812.933 bytes). O resultado fica em cache em `outputs/temporal/inspecao_schema_sinasc.json`; esta célula só refaz a consulta com `REATUALIZAR_INSPECAO = True`.

In [3]:
import requests
campos = sorted({c for grupo in tp.VARIAVEIS_CRITICAS.values() for c in grupo} | {'VERSAOSIST', 'ORIGEM'})
if cache_inspecao.exists() and not REATUALIZAR_INSPECAO:
    registro = json.loads(cache_inspecao.read_text(encoding='utf-8'))
else:
    sessao = requests.Session(); anos = {}
    for ano in ANOS_CANDIDATOS:
        url = tp.url_sinasc(ano)
        cabecalho = sessao.head(url, timeout=30)
        item = {'http_status': cabecalho.status_code, 'ultima_modificacao': cabecalho.headers.get('Last-Modified')}
        if cabecalho.ok:
            item.update(tp.inspecionar_zip_remoto(url, campos_dominio=campos, sessao=sessao))
        anos[str(ano)] = item
    registro = {'fonte': tp.PAGINA_OFICIAL, 'padrao_url': tp.URL_SINASC_CSV,
                'consultado_em': date.today().isoformat(), 'metodo': 'HEAD + HTTP Range; amostra = prefixo do CSV',
                'status_oficial_observado': {str(k): v for k, v in tp.STATUS_OFICIAL_OBSERVADO.items()},
                'status_consultado_em': tp.DATA_CONSULTA_STATUS, 'anos': anos}
    pasta_temporal.mkdir(parents=True, exist_ok=True)
    cache_inspecao.write_text(json.dumps(registro, ensure_ascii=False, indent=1) + '\n', encoding='utf-8')

inspecoes = {int(a): v for a, v in registro['anos'].items() if v.get('http_status') == 200}
disponibilidade = pd.DataFrame([{
    'ano': a, 'disponivel': a in inspecoes,
    'status_oficial': tp.STATUS_OFICIAL_OBSERVADO.get(a, 'definitivo (sem qualificador)'),
    'formato': 'ZIP contendo CSV' if a in inspecoes else None,
    'zip_mb': round(inspecoes[a]['bytes_zip'] / 1e6, 1) if a in inspecoes else None,
    'csv_mb': round(inspecoes[a]['bytes_csv_descomprimido'] / 1e6, 1) if a in inspecoes else None,
    'encoding': inspecoes.get(a, {}).get('encoding'), 'separador': inspecoes.get(a, {}).get('separador'),
    'n_colunas': len(inspecoes[a]['colunas']) if a in inspecoes else None,
    'linhas_na_amostra': inspecoes.get(a, {}).get('n_linhas_amostra'),
    'ultima_modificacao': registro['anos'][str(a)].get('ultima_modificacao'),
} for a in ANOS_CANDIDATOS])
print('Consulta remota em:', registro['consultado_em'])
display(disponibilidade)

Consulta remota em: 2026-09-29


,ano,disponivel,status_oficial,formato,zip_mb,csv_mb,encoding,separador,n_colunas,linhas_na_amostra,ultima_modificacao
0,1996,True,definitivo (sem qualificador),ZIP contendo CSV,32.3,314.0,utf-8,;,21,136458,"Fri, 08 May 2026 17:12:59 GMT"
1,1997,True,definitivo (sem qualificador),ZIP contendo CSV,36.9,350.3,utf-8,;,23,129642,"Fri, 08 May 2026 17:13:06 GMT"
2,1998,True,definitivo (sem qualificador),ZIP contendo CSV,37.7,346.5,utf-8,;,21,131411,"Fri, 08 May 2026 17:13:13 GMT"
3,1999,True,definitivo (sem qualificador),ZIP contendo CSV,40.2,382.9,utf-8,;,23,110251,"Fri, 08 May 2026 17:13:20 GMT"
4,2000,True,definitivo (sem qualificador),ZIP contendo CSV,41.2,388.1,utf-8,;,24,121231,"Fri, 08 May 2026 17:13:27 GMT"
5,2001,True,definitivo (sem qualificador),ZIP contendo CSV,44.6,401.5,utf-8,;,23,107501,"Tue, 21 Oct 2025 20:09:29 GMT"
6,2002,True,definitivo (sem qualificador),ZIP contendo CSV,44.2,410.5,utf-8,;,24,108102,"Tue, 21 Oct 2025 20:09:29 GMT"
7,2003,True,definitivo (sem qualificador),ZIP contendo CSV,44.9,411.5,utf-8,;,24,107849,"Tue, 21 Oct 2025 20:09:29 GMT"
8,2004,True,definitivo (sem qualificador),ZIP contendo CSV,44.2,409.6,utf-8,;,24,107724,"Tue, 21 Oct 2025 20:09:29 GMT"
9,2005,True,definitivo (sem qualificador),ZIP contendo CSV,44.3,411.7,utf-8,;,24,107553,"Tue, 21 Oct 2025 20:09:29 GMT"


Todos os anos de 1996 a 2026 estão publicados. O portal qualifica **2025 como "preliminar"** e **2026 como "1ª prévia"** (consultado em 29/09/2026); os demais não têm qualificador. Os arquivos de 2016–2024 foram republicados em 08/05/2026: a proveniência de cada ano precisa de hash próprio, porque o arquivo pode mudar entre consultas.

## 3. Comparabilidade de esquema

Comparação por **nome** (sem diferenciar caixa) com o esquema de 2024, e presença das variáveis essenciais ao desenho congelado. A classificação aplica os critérios de `tp.classificar_ano`, em ordem de gravidade:

1. sem `MESPRENAT` ou `PESO` → `INCOMPATIVEL_SEM_T_OU_Y`;
2. sem alguma covariável essencial → `SENSIBILIDADE_PENDENTE_COVARIAVEIS_AUSENTES`;
3. arquivo não consolidado (preliminar/prévia) → `EXCLUIDO_DADO_NAO_CONSOLIDADO`;
4. caso contrário → `JANELA_PRINCIPAL`.

In [4]:
compat = pd.DataFrame(tp.resumir_compatibilidade(inspecoes))
compat['classe'] = [tp.classificar_ano(l, tp.STATUS_OFICIAL_OBSERVADO.get(l['ano'])) for l in compat.to_dict('records')]
compat['status_oficial'] = [tp.STATUS_OFICIAL_OBSERVADO.get(a, 'definitivo') for a in compat.ano]
resumo = compat[['ano', 'n_colunas', 'tratamento_MESPRENAT', 'outcome_PESO', 'essenciais_ausentes',
                 'ausentes_vs_2024', 'extras_vs_2024', 'classe']].copy()
resumo['ausentes_vs_2024'] = resumo.ausentes_vs_2024.map(lambda l: f'{len(l)}: ' + ', '.join(l[:6]) + (' …' if len(l) > 6 else ''))
display(resumo)
(pasta_temporal / 'compatibilidade_anos.json').write_text(
    json.dumps(compat.to_dict('records'), ensure_ascii=False, indent=1) + '\n', encoding='utf-8')

,ano,n_colunas,tratamento_MESPRENAT,outcome_PESO,essenciais_ausentes,ausentes_vs_2024,extras_vs_2024,classe
0,1996,21,False,True,"[ESCMAE2010, MESPRENAT, PARIDADE, RACACORMAE]","41: CODESTAB, CODMUNNATU, CODPAISRES, CODUFNAT...",[],INCOMPATIVEL_SEM_T_OU_Y
1,1997,23,False,True,"[ESCMAE2010, MESPRENAT, PARIDADE, RACACORMAE]","39: CODESTAB, CODMUNNATU, CODPAISRES, CODUFNAT...",[],INCOMPATIVEL_SEM_T_OU_Y
2,1998,21,False,True,"[ESCMAE2010, MESPRENAT, PARIDADE, RACACORMAE]","41: CODESTAB, CODMUNNATU, CODPAISRES, CODUFNAT...",[],INCOMPATIVEL_SEM_T_OU_Y
3,1999,23,False,True,"[ESCMAE2010, MESPRENAT, PARIDADE, RACACORMAE]","39: CODESTAB, CODMUNNATU, CODPAISRES, CODUFNAT...",[],INCOMPATIVEL_SEM_T_OU_Y
4,2000,24,False,True,"[ESCMAE2010, MESPRENAT, PARIDADE, RACACORMAE]","39: CODMUNNATU, CODPAISRES, CODUFNATU, CONSPRE...",[UFINFORM],INCOMPATIVEL_SEM_T_OU_Y
5,2001,23,False,True,"[ESCMAE2010, MESPRENAT, PARIDADE, RACACORMAE]","39: CODMUNNATU, CODPAISRES, CODUFNATU, CONSPRE...",[],INCOMPATIVEL_SEM_T_OU_Y
6,2002,24,False,True,"[ESCMAE2010, MESPRENAT, PARIDADE, RACACORMAE]","39: CODMUNNATU, CODPAISRES, CODUFNATU, CONSPRE...",[UFINFORM],INCOMPATIVEL_SEM_T_OU_Y
7,2003,24,False,True,"[ESCMAE2010, MESPRENAT, PARIDADE, RACACORMAE]","39: CODMUNNATU, CODPAISRES, CODUFNATU, CONSPRE...",[UFINFORM],INCOMPATIVEL_SEM_T_OU_Y
8,2004,24,False,True,"[ESCMAE2010, MESPRENAT, PARIDADE, RACACORMAE]","39: CODMUNNATU, CODPAISRES, CODUFNATU, CONSPRE...",[UFINFORM],INCOMPATIVEL_SEM_T_OU_Y
9,2005,24,False,True,"[ESCMAE2010, MESPRENAT, PARIDADE, RACACORMAE]","39: CODMUNNATU, CODPAISRES, CODUFNATU, CONSPRE...",[UFINFORM],INCOMPATIVEL_SEM_T_OU_Y


22692

**Quebras estruturais identificadas:**

- **1996–2009:** não existe `MESPRENAT` (o mês de início do pré-natal só entra com a nova Declaração de Nascido Vivo). O tratamento não é construível: anos incompatíveis. Entre 2005 e 2006 muda também o separador (`;` para `,`) e o layout (21–24 para 30 colunas).
- **2010–2013 (transição da nova DN):** `MESPRENAT` e `PESO` existem, mas falta `PARIDADE` (2010–2013) e `ESCMAE2010` (2010–2011); `CONSPRENAT` falta em 2013. O X congelado não é replicável sem uma regra derivada nova.
- **2014 em diante:** esquema de 2024 completo, exceto `OPORT_DN` (sem papel no desenho, ausente até 2022). É o primeiro ano em que as sete covariáveis, T, Y e os auxiliares coexistem.

## 4. Variáveis críticas do desenho de 2024

| Grupo | Campos brutos | Uso no desenho certificado |
|---|---|---|
| A. Desfecho | `PESO` | Y = `PESO` < 2.500 g; qualidade P1 = 500–6.000 g |
| B. Tratamento | `MESPRENAT` | T = 1 para meses 1–3; 0 para 4–9; 99/vazio excluídos |
| C. Covariáveis pré-tratamento | `IDADEMAE`, `ESCMAE2010`, `RACACORMAE`, `ESTCIVMAE`, `PARIDADE`, `QTDFILMORT` | X congelado; 9/99 viram categoria IGNORADO |
| D. Auxiliares | `GRAVIDEZ`, `CONSPRENAT`, `DTNASC`, `CONTADOR` | gestação única; sensibilidade `CONSPRENAT=0`; ano de `DTNASC`; chave |
| E. Geográficas | `CODMUNRES` | UF de residência (X) e município (agrupamentos, partições) |
| F. Somente auditoria | `CONSULTAS`, `SEMAGESTAC`, `GESTACAO`, `PARTO`, `KOTELCHUCK`, `ESCMAE`, `QTDGESTANT` | diagnóstico; proibidas no X (pós-tratamento/mediadoras) |

In [5]:
linhas = []
for grupo, campos_grupo in tp.VARIAVEIS_CRITICAS.items():
    for campo in campos_grupo:
        linha = {'grupo': grupo, 'campo': campo}
        for a in range(2010, 2027):
            linha[a] = '✓' if campo in tp.normalizar_colunas(inspecoes[a]['colunas']) else '—'
        linhas.append(linha)
display(pd.DataFrame(linhas).set_index(['grupo', 'campo']))

2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 2025 2026
grupo                        campo                                                                                          
A_outcome                    PESO          ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
B_tratamento                 MESPRENAT     ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
C_covariaveis_pre_tratamento IDADEMAE      ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
                             ESCMAE2010    —    —    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
                             RACACORMAE    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
                             ESTCIVMAE     ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
                             PARIDADE      —    —    —    —    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
                             QTDFILMORT    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
D_auxiliares                 GRAVIDEZ      ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
                             CONSPRENAT    ✓    ✓    ✓    —    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
                             DTNASC        ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
                             CONTADOR      ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
E_geograficas                CODMUNRES     ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
F_auditoria_diagnostico      CONSULTAS     ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
                             SEMAGESTAC    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
                             GESTACAO      ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
                             PARTO         ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
                             KOTELCHUCK    —    —    —    —    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
                             ESCMAE        ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
                             QTDGESTANT    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓

## 5. Codificação das variáveis críticas

O mesmo nome não garante a mesma semântica. Abaixo, os **códigos observados** nas amostras de 2010 a 2026, comparados aos de 2024.

Ressalva importante: a amostra é o **início físico** de cada arquivo (em geral ordenado por lote/UF, ≈35–40 mil registros). Serve para detectar códigos novos, removidos ou reformatados; não serve para frequências nem prevalências, que ficam para o notebook 07 com os arquivos completos.

In [6]:
categoricas = ['MESPRENAT', 'GRAVIDEZ', 'ESCMAE2010', 'RACACORMAE', 'ESTCIVMAE', 'PARIDADE']
def codigos(ano, campo):
    d = inspecoes[ano]['dominios_amostra'].get(campo)
    if d is None:
        return None
    return sorted('ausente' if k in ('<VAZIO>', 'NA') else k for k in d)
tabela = {campo: {a: codigos(a, campo) for a in range(2010, 2027)} for campo in categoricas}
exibicao = pd.DataFrame({c: {a: ('AUSENTE' if v is None else ','.join(v)) for a, v in anos.items()} for c, anos in tabela.items()})
display(exibicao)
mudancas = [(c, a, sorted(set(v) ^ set(tabela[c][2024]))) for c, anos in tabela.items()
            for a, v in anos.items() if a >= 2014 and v is not None and set(v) != set(tabela[c][2024])]
print('Diferenças de códigos em 2014–2026 frente a 2024 (na amostra):', mudancas or 'nenhuma')

,MESPRENAT,GRAVIDEZ,ESCMAE2010,RACACORMAE,ESTCIVMAE,PARIDADE
2010,"01,02,03,04,05,06,07,08,09,10,99,ausente","1,2,3,9,ausente",AUSENTE,"1,2,3,4,5,9,ausente","1,2,3,4,5,9,ausente",AUSENTE
2011,"01,02,03,04,05,06,07,08,09,99,ausente","1,2,3,9,ausente",AUSENTE,"1,2,3,4,5,ausente","1,2,3,4,5,9,ausente",AUSENTE
2012,"01,02,03,04,05,06,07,08,09,99,ausente","1,2,3,9,ausente","0,1,2,3,4,5,9,ausente","1,2,3,4,5,ausente","1,2,3,4,5,9,ausente",AUSENTE
2013,"01,02,03,04,05,06,07,08,09,99,ausente","1,2,3,9,ausente","0,1,2,3,4,5,9,ausente","1,2,3,4,5,ausente","1,2,3,4,5,9,ausente",AUSENTE
2014,"01,02,03,04,05,06,07,08,09,10,99,ausente","1,2,3,9,ausente","0,1,2,3,4,5,9,ausente","1,2,3,4,5,9,ausente","1,2,3,4,5,9,ausente","0,1"
2015,"01,02,03,04,05,06,07,08,09,99,ausente","1,2,3,9,ausente","0,1,2,3,4,5,9,ausente","1,2,3,4,5,ausente","1,2,3,4,5,9,ausente","0,1"
2016,"01,02,03,04,05,06,07,08,09,99,ausente","1,2,3,9,ausente","0,1,2,3,4,5,9,ausente","1,2,3,4,5,ausente","1,2,3,4,5,9,ausente","0,1"
2017,"01,02,03,04,05,06,07,08,09,99,ausente","1,2,3,9,ausente","0,1,2,3,4,5,9,ausente","1,2,3,4,5,ausente","1,2,3,4,5,9,ausente","0,1"
2018,"01,02,03,04,05,06,07,08,09,99,ausente","1,2,3,9,ausente","0,1,2,3,4,5,9,ausente","1,2,3,4,5,ausente","1,2,3,4,5,9,ausente","0,1"
2019,"01,02,03,04,05,06,07,08,09,99,ausente","1,2,3,9,ausente","0,1,2,3,4,5,9,ausente","1,2,3,4,5,ausente","1,2,3,4,5,9,ausente","0,1"


Diferenças de códigos em 2014–2026 frente a 2024 (na amostra): [('MESPRENAT', 2014, ['10']), ('RACACORMAE', 2014, ['9'])]


Leitura:

- De **2014 a 2026** os códigos de T, gestação, escolaridade 2010, raça/cor, situação conjugal e paridade coincidem com 2024. As únicas diferenças na amostra são o código `10` de `MESPRENAT` (fora do domínio 1–9; excluído pela regra de T, como em 2024) e o `9` de `RACACORMAE` em 2014 (já tratado como IGNORADO). O resultado precisa ser confirmado nos arquivos completos.
- `PARIDADE` só assume 0/1 em todos os anos em que existe; o dicionário oficial disponível (1996–2019) não documenta esse campo: questão aberta herdada de 2024.
- O código `99` de `MESPRENAT` aparece em todos os anos e continua sem definição no dicionário estrutural 1996–2019; em 2024 foi interpretado como ignorado com base no manual da DNV de 2022.
- Em **2010–2013** o valor ausente vem gravado como o texto literal `NA`; a partir de 2014, como campo vazio. Ambos viram nulo na leitura, mas isso precisa ser explícito.

## 6. Diferenças de formato entre anos

In [7]:
fmt = []
for a in range(2010, 2027):
    colunas = inspecoes[a]['colunas']; nomes = tp.normalizar_colunas(colunas)
    fmt.append({'ano': a, 'separador': inspecoes[a]['separador'],
                'nome_da_chave': nomes.get('CONTADOR'),
                'posicao_da_chave': colunas.index(nomes['CONTADOR']) + 1 if 'CONTADOR' in nomes else None,
                'nomes_em_minusculas': sum(c.islower() for c in colunas),
                'coluna_sem_nome': '' in colunas,
                'ausente_como_texto_NA': any('NA' in (d or {}) for d in inspecoes[a]['dominios_amostra'].values()),
                'versoes_do_sistema': ','.join(sorted(k for k in inspecoes[a]['dominios_amostra'].get('VERSAOSIST') or {} if k not in ('<VAZIO>', 'NA')))})
display(pd.DataFrame(fmt))

,ano,separador,nome_da_chave,posicao_da_chave,nomes_em_minusculas,coluna_sem_nome,ausente_como_texto_NA,versoes_do_sistema
0,2010,",",contador,2,1,True,True,"1.9.00,2.2.00,2.2.07,2.2.08,3.2.00,3.2.01"
1,2011,",",contador,2,1,True,True,"3.2.00,3.2.01"
2,2012,",",contador,2,1,True,True,"3.2.00,3.2.01"
3,2013,",",contador,2,1,True,True,"3.2.00,3.2.01"
4,2014,;,CONTADOR,1,0,False,False,"3.2.00,3.2.01"
5,2015,;,CONTADOR,1,0,False,False,"3.2.00,3.2.01,3.2.02"
6,2016,;,contador,61,61,False,False,"3.2.00,3.2.01,3.2.02"
7,2017,;,CONTADOR,1,0,False,False,"3.2.00,3.2.01,3.2.02"
8,2018,;,CONTADOR,61,0,False,False,"3.2.00,3.2.01,3.2.02"
9,2019,;,CONTADOR,61,0,False,False,"2.2.03,3.2.00,3.2.01,3.2.02"


Consequências para o carregamento multianual:

- **nunca ler por posição:** a chave `CONTADOR` é a 1ª coluna em 2014, 2015, 2017 e 2023–2026 e a última em 2016 e 2018–2022;
- **não diferenciar caixa:** 2016 tem todos os nomes em minúsculas; a chave aparece como `contador` ou `CONTADOR`. O `converter_zip_para_parquet` histórico exige nomes exatos; `tp.converter_sinasc_ano` passa os nomes originais do próprio ano;
- **aspas variam** entre anos (2024 e 2016 sem aspas; os demais com aspas), o que explica o CSV de 2024 menor que os vizinhos; não altera valores;
- 2006–2013 têm uma coluna sem nome (vírgula final); 2010 mistura versões antigas do sistema (1.9 e 2.2) com a 3.2, marca da transição;
- `CONTADOR` é identificador técnico **dentro** do arquivo: a chave multianual precisa ser `(ano, CONTADOR)`, sem supor ligação entre anos.

## 7. Riscos de comparabilidade substantiva

Nem todo risco aparece no esquema:

1. **COVID-19 (2020–2021):** acesso ao pré-natal e perfil de nascimentos alterados; não é motivo de exclusão, mas é candidato natural a heterogeneidade temporal.
2. **Zika e microcefalia (2015–2016):** mudança de comportamento reprodutivo e de vigilância em algumas UFs.
3. **Queda da natalidade e mudança de composição:** a população de nascidos vivos muda ano a ano; diferenças de estimativa podem refletir composição, não efeito.
4. **Qualidade do preenchimento:** proporções de ignorado e vazio em `MESPRENAT`, `ESCMAE2010`, `PARIDADE` e `RACACORMAE` podem variar por ano e UF; a elegibilidade (T conhecido) é seletiva e sua seletividade pode mudar.
5. **Documentação:** o dicionário oficial disponível cobre 1996–2019; não há layout oficial por ano para 2020–2024.
6. **Versão do arquivo:** republicações (2016–2024 em 08/05/2026) exigem hash por ano; resultados só são comparáveis à versão registrada.
7. **Arquivos não consolidados:** 2025 (preliminar) e 2026 (prévia parcial, ≈⅓ do tamanho típico) ainda podem receber registros e correções.

## 8. Janela temporal

In [8]:
janela = compat.groupby('classe').ano.apply(lambda s: f'{s.min()}–{s.max()} ({len(s)} anos)' if len(s) > 1 else str(s.iloc[0]))
display(janela.to_frame('anos'))
principal = compat.loc[compat.classe == 'JANELA_PRINCIPAL', 'ano'].tolist()
assert principal == list(range(2014, 2025)) and tp.ANO_BASELINE in principal
novos = [a for a in principal if a != tp.ANO_BASELINE]
download_mb = sum(inspecoes[a]['bytes_zip'] for a in novos) / 1e6
print(f'Janela principal: {principal[0]}–{principal[-1]} ({len(principal)} anos; {len(novos)} anos novos)')
print(f'Download necessário para a próxima rodada: {download_mb:,.0f} MB comprimidos (2014–2023)')
print(f'Ordem de grandeza do Parquet: ~{c2024.parquet.stat().st_size / 1e6:.0f} MB por ano (referência 2024)')

,anos
classe,
EXCLUIDO_DADO_NAO_CONSOLIDADO,2025–2026 (2 anos)
INCOMPATIVEL_SEM_T_OU_Y,1996–2009 (14 anos)
JANELA_PRINCIPAL,2014–2024 (11 anos)
SENSIBILIDADE_PENDENTE_COVARIAVEIS_AUSENTES,2010–2013 (4 anos)


Janela principal: 2014–2024 (11 anos; 10 anos novos)
Download necessário para a próxima rodada: 1,147 MB comprimidos (2014–2023)
Ordem de grandeza do Parquet: ~56 MB por ano (referência 2024)


**Janela principal proposta: 2014–2024 (11 anos).**

Justificativa:

- é a **maior janela** em que T, Y, as sete covariáveis congeladas, `GRAVIDEZ`, `CONSPRENAT`, `DTNASC` e o município coexistem com os **mesmos códigos** de 2024;
- 2014 é o primeiro ano após a transição da nova DN (2010–2013), período com layout, versões do sistema e representação de ausentes instáveis;
- todos os anos da janela estão publicados sem qualificador de preliminar;
- custo viável: ≈1,15 GB comprimidos e ~0,5–0,6 GB em Parquet para os 10 anos novos.

Fora da janela principal:

| Anos | Classe | Uso possível |
|---|---|---|
| 1996–2009 | incompatível | nenhum para este desenho (sem T) |
| 2010–2013 | sensibilidade pendente | só com regra de paridade derivada (ex.: `QTDPARTNOR`+`QTDPARTCES`) e escolaridade alternativa, **documentadas antes** de ver resultados; nunca misturada à janela principal |
| 2025 | não consolidado | reavaliar quando o portal remover "preliminar" |
| 2026 | prévia parcial | excluído |

Esta é uma decisão **provisória**: ela depende da confirmação, nos arquivos completos, dos critérios da seção 9.

## 9. Critérios para admitir um ano na análise

Um ano entra na análise temporal somente se **todos** forem satisfeitos:

1. arquivo oficial baixado com SHA-256 registrado em manifesto próprio (`outputs/temporal/manifestos/`);
2. esquema com todas as variáveis essenciais (verificado por nome, sem diferenciar caixa) — **atendido em 2014–2024 nesta auditoria**;
3. códigos de T, gestação e das covariáveis categóricas contidos no domínio de 2024 — **atendido na amostra; pendente nos arquivos completos**;
4. `DTNASC` válida e pertencente ao ano do arquivo em todos os registros (a regra de 2024 exige zero datas fora do ano);
5. `CONTADOR` completo e único dentro do ano;
6. `CODMUNRES` com 6 dígitos em todos os registros elegíveis;
7. fluxo A0–A3 reconciliável e com ambos os grupos de T e ambos os valores de Y;
8. proporção de T desconhecido e de categorias IGNORADO registrada e comparada com 2024 (tolerâncias a fixar no notebook 07, antes de qualquer estimativa);
9. arquivo oficialmente consolidado.

Falha nos itens 1–7 exclui o ano; o item 8 não exclui automaticamente, mas precisa ser explicado.

## 10. Desenho temporal em camadas

- **Camada A — auditoria temporal (notebook 07):** N por ano, fluxo A0–A3, ausentes/ignorados, prevalência de Y, proporção de T, distribuição das covariáveis, geografia por UF.
- **Camada B — comparabilidade (notebook 08):** SMD por ano, mudanças de composição frente a 2024, escore de propensão e suporte comum por ano, positividade por perfis, com o mesmo X e o mesmo modelo de 2024.
- **Camada C — estabilidade das estimativas (notebook 09):** AIPW **independente por ano**, com as especificações C1/C2 congeladas; comparação com 2024 como referência, sem reajustar 2024.
- **Camada D — análise empilhada (eventual):** somente depois das camadas A–C, se a estrutura temporal justificar (efeitos fixos de ano, UF×ano, inferência agrupada por município). **O empilhamento não será automaticamente a análise principal.**

## 11. Pendências

- baixar e converter 2014–2023 (≈1,15 GB): autorização explícita na próxima rodada;
- verificar os critérios 3–8 nos arquivos completos (notebook 07);
- fixar tolerâncias de qualidade antes de ver estimativas;
- decidir se 2010–2013 terão análise de sensibilidade e com qual regra derivada;
- reconsultar o status oficial de 2025;
- carregamento multianual da amostra: a função histórica `carregar_amostra` exige datas de 2024 e **não** será alterada; a versão por ano será nova, em `src/temporal.py`.

## Status

`BASELINE_2024_PRESERVADO` · `AUDITORIA_TEMPORAL_INICIADA_COM_SUCESSO` · janela principal provisória 2014–2024 · nenhuma estimativa multianual produzida.